In [0]:
%pip install azure-identity
%pip install azure-keyvault-secrets
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window
from pytz import timezone
from datetime import date, timedelta, datetime
import time
import os
import json
import logging
from delta.tables import *


Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.1/186.1 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.4/207.4 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-dbc9ca02-03a1-4265-a8e7-28ca9620eab5
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.

In [0]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
# MAGIC %run ./utils/jdbc_utils


In [0]:
def get_arguments():
    try:
        print("Get passed arguments Starting...")
        DatabricksEnv = getArgument(f"DatabricksEnv")
        loadtracker_target_catalog = getArgument(f"monitoringcatalog")
        loadtracker_target_schema = getArgument(f"monitoringschema")
        loadtracker_target_table = getArgument(f"monitoringtable")
        efs_file_dict_jsonstring = getArgument(f"dict_sql")
        print("Passed arguments Completed...")


        print("Map json string to dictionary Starting...")
        efs_file_dict = json.loads(efs_file_dict_jsonstring)
        print("Map json string to dictionary Completed...")

        print("Map dictionary values to runtime variables Starting...")
        entity_name = efs_file_dict['Entity']
        time_grain = efs_file_dict['Time_Grain']
        incremental_load_flag = efs_file_dict['Incremental_Load_Flag']
        source_incremental_identifier_col = efs_file_dict['SourceIncremental_Identifier_Col']
        uc_incremental_identifier_col = efs_file_dict['UCIncremental_Identifier_Col']
        target_file_path = efs_file_dict['Target_File_Path']
        target_file_format = efs_file_dict['Target_File_Format']
        target_entity = efs_file_dict['Target_Entity']
        target_uc_schema = efs_file_dict['Target_UC_Schema']
        jdbc_sql = efs_file_dict['SQL'][DatabricksEnv]
        target_unity_catalog =  efs_file_dict['Target_Unity_Catalog'][DatabricksEnv]
        secondary_unity_catalog =  efs_file_dict['Secondary_Unity_Catalog'][DatabricksEnv]
        target_datalake =  efs_file_dict['Target_Datalake'][DatabricksEnv]
        secondary_datalake =  efs_file_dict['Secondary_Datalake'][DatabricksEnv]
        dbtype = efs_file_dict['DBType']
        driver = efs_file_dict.get('Driver', '')
        primarykeys = efs_file_dict['PrimaryKey']
        dedupe_key_columns = efs_file_dict.get('Dedupe_Key_Columns', primarykeys)
        dedupe_order_columns = efs_file_dict.get('Dedupe_Order_Columns', 'UpdatedDateTime,CreatedDateTime')
        raw_merge_on_primary_key = str(efs_file_dict.get('Raw_Merge_On_Primary_Key', 'N')).upper()

        source_connection_config = None
        source_connection_config_path = efs_file_dict.get('Source_Connection_Config')
        if source_connection_config_path:
            config_path = source_connection_config_path if source_connection_config_path.startswith('./') else f"./{source_connection_config_path}"
            source_connection_config = load_json_config(config_path)

        dburl = ''
        keyvault = ''
        password = ''
        user = ''
        if not is_azure_sql_db_type(dbtype):
            dburl = efs_file_dict['URL'][DatabricksEnv]
            keyvault = efs_file_dict['keyvault'][DatabricksEnv]
            password = efs_file_dict['PasswordSecret'][DatabricksEnv]
            user = efs_file_dict['NHA'][DatabricksEnv]
              
        print("Map dictionary values to runtime variables Completed...")

        return {
            "DatabricksEnv": DatabricksEnv,
            "loadtracker_target_catalog": loadtracker_target_catalog,
            "loadtracker_target_schema": loadtracker_target_schema,
            "loadtracker_target_table": loadtracker_target_table,
            "entity_name": entity_name,
            "time_grain": time_grain,
            "incremental_load_flag": incremental_load_flag,
            "source_incremental_identifier_col": source_incremental_identifier_col,
            "uc_incremental_identifier_col": uc_incremental_identifier_col,
            "target_file_path": target_file_path,
            "target_file_format": target_file_format,
            "target_entity": target_entity,
            "target_uc_schema": target_uc_schema,
            "jdbc_sql": jdbc_sql,
            "target_unity_catalog": target_unity_catalog,
            "secondary_unity_catalog": secondary_unity_catalog,
            "target_datalake": target_datalake,
            "secondary_datalake": secondary_datalake,
            "dburl": dburl,
            "driver": driver,
            "dbtype": dbtype,
            "keyvault": keyvault,
            "password": password,
            "user": user,
            "primarykeys": primarykeys,
            "dedupe_key_columns": dedupe_key_columns,
            "dedupe_order_columns": dedupe_order_columns,
            "raw_merge_on_primary_key": raw_merge_on_primary_key,
            "source_connection_config": source_connection_config
        }
    except Exception as e:
        logger.error(f"Error: Failed to get_arguments - {str(e)}")
        raise Exception("Error: Failed to get_arguments - " + str(e))

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

In [0]:
def get_current_time():
    try:
        tz = timezone(f"EST")
        DATE = datetime.now(tz)

        return DATE
    except Exception as e:
        logger.error(f"Error: Failed to get_current_time - {str(e)}")
        raise Exception("Error: Failed to get_current_time - " + str(e))

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
def get_last_load_timestamp():
    try:

        
        _tracker_select_sql = f""" SELECT MAX(LAST_LOAD_TIMESTAMP)  FROM {args['loadtracker_target_catalog']}.{args['loadtracker_target_schema']}.{args['loadtracker_target_table']} WHERE LOAD_COMPLETE_FLAG ='Y' AND TABLE_NAME ='{args['entity_name']}' """

        logger.info(f"Last Load Timestamp SQL Statement: {_tracker_select_sql}")
        extract_load_timestamp = (spark.sql(_tracker_select_sql)).first()[0]
        if extract_load_timestamp is None:
            targetFullEntity = f"{args['target_unity_catalog']}.{args['target_uc_schema']}.{args['target_entity']}"
            _tracker_select_sql = f""" SELECT MAX((to_timestamp({args['uc_incremental_identifier_col']},'yyyy-MM-dd HH:mm:ss.SSSSSS'))) FROM {targetFullEntity}"""
            logger.info(f"Last Load SQL: {_tracker_select_sql}")
            extract_load_timestamp = (spark.sql(_tracker_select_sql)).first()[0]
        


        logger.info(f"Last Load Timestamp: {extract_load_timestamp}")
        return extract_load_timestamp
    except Exception as e:
         raise Exception("Error: Failed to get_last_load_timestamp - "+str(e))


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
def does_table_exist(args):    
    try:
        target_unity_catalog = args['target_unity_catalog']
        target_uc_schema = args['target_uc_schema']
        target_entity = args['target_entity']

        # Check if the table exists in the Unity Catalog
        table_exists = target_entity.lower() in spark.sql(f"SHOW TABLES IN {target_unity_catalog}.{target_uc_schema}").toPandas()['tableName'].str.lower().values
        return table_exists
    except Exception as e:
        logger.error(f"Error: Failed to check if table exists - {str(e)}")
        raise Exception("Error: Failed to check if table exists - " + str(e))

In [0]:
def does_transform_table_exist(args):    
    try:
        target_unity_catalog = args['target_unity_catalog']
        #replace raw with transform
        target_uc_schema = args['target_uc_schema'].replace("raw","transform")
        target_entity = args['target_entity']

        # Check if the table exists in the Unity Catalog
        table_exists = target_entity.lower() in spark.sql(f"SHOW TABLES IN {target_unity_catalog}.{target_uc_schema}").toPandas()['tableName'].str.lower().values
        return table_exists
    except Exception as e:
        logger.error(f"Error: Failed to check if table exists - {str(e)}")
        raise Exception("Error: Failed to check if table exists - " + str(e))

In [0]:
def create_sql_statement(args):
    try:
        if args['incremental_load_flag'] == 'Y':
            if args['time_grain'] == 'HOURLY' or args['time_grain'] == 'DAILY':
                _time = get_last_load_timestamp()
                logger.info(f"Last load timestamp: {_time}")
                if args['dbtype']=='Oracle':
                    sqlstmt = f'''{args['jdbc_sql']} AND {args['source_incremental_identifier_col']} > to_timestamp('{_time}','YYYY-MM-DD HH24:MI:SS.ff6') '''
                elif is_azure_sql_db_type(args['dbtype']):
                    watermark = format_sql_server_watermark(_time)
                    sqlstmt = f"{args['jdbc_sql']} AND {args['source_incremental_identifier_col']} > CAST('{watermark}' AS datetime2)"
                else:
                    sqlstmt = f'''{args['jdbc_sql']} AND {args['source_incremental_identifier_col']} > timestamp_format('{_time}','YYYY-MM-DD HH24:MI:SS.ff6') '''
            else:
                raise ValueError("Invalid time_grain value. Expected 'HOURLY' or 'DAILY'.")
        else:
            # Non-incremental load: Use the base query without filtering
            sqlstmt = args['jdbc_sql']

        # Log the generated SQL statement for debugging
        sqlstmt = "(" + sqlstmt + ")"
        logger.info(f"Generated SQL Statement: {sqlstmt}")
        return sqlstmt
    except Exception as e:
        logger.exception("Error: Failed to create SQL statement")
        raise Exception("Error: Failed to create SQL statement - " + str(e))

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
def persist_new_data(args):
    try:
        # Define target paths and entities
        targetdatalake=args['target_file_path']
        secondarydatalake=args['secondary_datalake']
        targetFullFilePath = f"{args['target_datalake']}{args['target_file_path']}{args['target_entity']}/{args['target_file_format']}"
        targetFullEntity = f"{args['target_unity_catalog']}.{args['target_uc_schema']}.{args['target_entity']}"
        transformFullEntity = f"{args['target_unity_catalog']}.{args['target_uc_schema'].replace('raw','transform')}.{args['target_entity']}"
        transformFullFilePath = targetFullFilePath.replace("Raw","Transform")
        secondaryFullFilePath = f"{args['secondary_datalake']}{args['target_file_path']}{args['target_entity']}/{args['target_file_format']}"
        secondaryFullEntity = f"{args['secondary_unity_catalog']}.{args['target_uc_schema']}.{args['target_entity']}"
        secondaryTransformFilePath = secondaryFullFilePath.replace("Raw","Transform")
        secondaryTransformEntity = f"{args['secondary_unity_catalog']}.{args['target_uc_schema'].replace('raw','transform')}.{args['target_entity']}"
        logger.info(f"SecondaryTransformFilePath: {secondaryTransformFilePath}")
        logger.info(f"SecondaryTransformEntity: {secondaryTransformEntity}")
       

        table_exists = does_table_exist(args)
        
        if table_exists:
            logger.info(f"Table {args['target_entity']} exists in {args['target_unity_catalog']}.{args['target_uc_schema']}.")
        else:
            logger.info(f"Table {args['target_entity']} does not exist in {args['target_unity_catalog']}.{args['target_uc_schema']}.")
            spark.sql(f"CREATE TABLE IF NOT EXISTS {targetFullEntity} OPTIONS (path '{targetFullFilePath}')")
        
        # Generate the SQL statement
        sqlstmt = create_sql_statement(args)
        logger.info(f"Executing SQL statement: {sqlstmt}")

        if is_azure_sql_db_type(args['dbtype']) or args['dbtype'] in ['Oracle','DB2']:
            remoteTableRawDF = read_jdbc_dataframe(spark, args, sqlstmt)
        else:
            remoteTableRawDF = spark.sql(sqlstmt)

        remoteTableRawDF = dedupe_dataframe(remoteTableRawDF, args)
        record_count = remoteTableRawDF.count()
        logger.info(f"Number of records to be inserted for entity {args['entity_name']}: {record_count}")

        # Check if the DataFrame is empty
        if record_count == 0:
            logger.info(f"No data returned for entity {args['entity_name']} {record_count} records found. Skipping persistence.")
            return record_count

        # Determine write mode
        write_mode = "append" if args['incremental_load_flag'] == 'Y' else "overwrite"
        primarykey = args['primarykeys'].split(',')
        merge_condition = build_merge_condition(primarykey)
        use_raw_merge = args.get('raw_merge_on_primary_key') == 'Y' and args['incremental_load_flag'] == 'Y' and table_exists

        # Write the data to the primary target table
        logger.info(f"Writing to {targetFullFilePath}")
        if use_raw_merge:
            logger.info(f"Merging raw Delta data using condition: {merge_condition}")
            merge_dataframe_into_delta(spark, remoteTableRawDF, targetFullFilePath, merge_condition)
        else:
            remoteTableRawDF.write.format("Delta").mode(write_mode).option("overwriteSchema", "true").saveAsTable(
                f"{targetFullEntity}", path=targetFullFilePath
            )

        # Clone the data to the secondary storage
        clone_sql = f"""
        CREATE OR REPLACE TABLE {secondaryFullEntity}
        DEEP CLONE {targetFullEntity}
        LOCATION '{secondaryFullFilePath}'
        """
        logger.info(f"Cloning data from {targetFullEntity} to {secondaryFullEntity}")
        spark.sql(clone_sql)

        # Log success message
        logger.info(f"Successfully persisted {record_count} records for entity {args['entity_name']} to {targetFullEntity} and cloned to {secondaryFullEntity}.")

        logger.info(f"Time Grain is {args['time_grain']}")

        transform_exists = does_transform_table_exist(args)
        
       
        if transform_exists:
            logger.info(f"Table {args['target_entity']} exists in {args['target_unity_catalog']}.{args['target_uc_schema'].replace('raw','transform')}.")
        else:
            logger.info(f"Table {args['target_entity']} does not exist in {args['target_unity_catalog']}.{args['target_uc_schema'].replace('raw','transform')}.")
            logger.info(f"transform full file path is '{transformFullFilePath}'")
            spark.sql(f"CREATE TABLE IF NOT EXISTS {transformFullEntity} LIKE {targetFullEntity} LOCATION '{transformFullFilePath}' USING DELTA")
            logger.info(f"Table {transformFullEntity} created.")
        if args['time_grain'] == 'DAILY':
           
            #copy to Transform
            remoteTableRawDF.write.format("Delta").mode(write_mode).option("overwriteSchema", "true").saveAsTable(f"{transformFullEntity}", path=transformFullFilePath)
            
        elif args['time_grain'] == 'HOURLY':
            if not transform_exists: #populate with full table query
                
                logger.info(f"{args['target_entity']} did not exist, so populating with full table query")
                sqlstmt = args['jdbc_sql']
                logger.info(f"New SQL statement: {sqlstmt}")
                if is_azure_sql_db_type(args['dbtype']) or args['dbtype'] in ['Oracle','DB2']:
                    sqlstmt = "(" + sqlstmt + ")"
                remoteTableRawDF = read_jdbc_dataframe(spark, args, sqlstmt)
                remoteTableRawDF = dedupe_dataframe(remoteTableRawDF, args)
                logger.info(f"new transform record count is {remoteTableRawDF.count()}")
                # Write the data to the primary transform table
                logger.info(f"Writing to {transformFullFilePath}")
                remoteTableRawDF.write.format("Delta").mode(write_mode).option("overwriteSchema", "true").saveAsTable(
                    f"{transformFullEntity}", path=transformFullFilePath
                )
            else:
                primarykey = args['primarykeys'].split(',')
                keystring = build_merge_condition(primarykey)
                logger.info(f"keystring={keystring}")
                merge_dataframe_into_delta(spark, remoteTableRawDF, transformFullFilePath, keystring)
            
        # Clone the data to the secondary storage
        clone_sql = f"""
        CREATE OR REPLACE TABLE {secondaryTransformEntity}
        DEEP CLONE {transformFullEntity}
        LOCATION '{secondaryTransformFilePath}'
        """
        logger.info(f"Cloning data from {transformFullEntity} to {secondaryTransformEntity}")
        logger.info(clone_sql)
        spark.sql(clone_sql)

        return record_count
    except Exception as e:
        logger.exception("Error: Failed to persist new data")
        raise Exception(f"Error: Failed to persist new data - {str(e)}")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
def update_monitor(args,recordsreturned):
    try:
        target_full_entity = f"{args['target_unity_catalog']}.{args['target_uc_schema']}.{args['target_entity']}"
        monitor_table = f"{args['loadtracker_target_catalog']}.{args['loadtracker_target_schema']}.{args['loadtracker_target_table']}"

        # Create the insert statement for the monitoring table
        if args['incremental_load_flag'] == 'Y':
            if args['time_grain'] == 'HOURLY' or args['time_grain'] == 'DAILY':
                #Get Max Update time
                monitor_sql = f"""SELECT Date_format(MAX(to_timestamp({args['uc_incremental_identifier_col']})),'yyyy-MM-dd HH:mm:ss.SSSSSS') FROM {target_full_entity}"""
                logger.info(f"Monitor sql: {monitor_sql}")
                _newwatermark = spark.sql(monitor_sql).collect()[0][0]
                #t = datetime.strptime(str(_newwatermark), '%Y-%m-%d %H:%M:%S.%f')
                insert_str = f"""
                    SELECT DISTINCT
                        '{args['entity_name']}' AS TABLE_NAME,
                        CURRENT_TIMESTAMP AS LOAD_DATE,
                        'Y' AS LOAD_COMPLETE_FLAG,
                        {recordsreturned} AS ROW_COUNT,
                        to_timestamp('{_newwatermark}','yyyy-MM-dd HH:mm:ss.SSSSSS')  AS LAST_LOAD_TIMESTAMP                        
                    FROM {target_full_entity}
                """ 

            else:
                raise ValueError("Invalid time_grain value. Expected 'CONTINUOUS', 'HOURLY' or 'DAILY'.")

            sql_update = f"INSERT INTO {monitor_table} {insert_str}"
            logger.info(f"Monitor sql update: {sql_update}")
            spark.sql(sql_update)

    except Exception as e:
        logger.exception("Error: Failed to update monitor")
        raise

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
try:
    args = get_arguments()
   

    recordsreturned = persist_new_data(args)
    logger.info(f"Records returned: {recordsreturned}")
    if recordsreturned > 0:
        logger.info(f"Updating monitor table data")
        update_monitor(args,recordsreturned)
except Exception as e:
    print("An error occurred:", str(e))
    raise e

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage